In [2]:
pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.6 MB 13.0 MB/s eta 0:00:01
   ------------------- -------------------- 4.7/9.6 MB 12.4 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.6 MB 14.5 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 14.5 MB/s  0:00:00
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   ---- ----------------------------------- 4.7/41.3 MB 22.0 MB/s eta 0:00:02
   ---------- ----------------------------- 10.7/41.3 MB 25.8 MB/s eta 0:00:02
   --------------- ------------------------ 16.0/41.3 MB 25.2 MB/s eta 0:00:02
   --------------------- ------------------ 22.0/41.3 MB 26.3 MB/s eta 0:00:01
   --------------------------- ------------ 28.8/41.3 MB 27.3 MB/s eta 0:00:01
   ----------------------------------- ---- 36.7/41.3 MB 29.2 MB/s eta 0:00:01
   ---------------------------------------- 41.3/41.3 MB 28.5 MB/s  0:00:01

   ---

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('icecreamsales.csv')

In [4]:
df

,Month,Sales
0,2019-01-31,412.92
1,2019-02-28,341.00
2,2019-03-31,324.47
3,2019-04-30,320.42
4,2019-05-31,277.35
5,2019-06-30,192.90
6,2019-07-31,259.00
7,2019-08-31,266.59
8,2019-09-30,297.42
9,2019-10-31,341.29


In [5]:
# convert Date string to datetime object
df['Month'] = pd.to_datetime(df['Month'], format = '%Y-%m-%d')

# assigning the results to all rows from the 'Date' column
# df.loc[:, 'Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# df2 = df.copy()
# df2['Date'] = pd.to_datetime(df2['Date'], format='%m/%d/%Y')
# df = df2

In [6]:
# create time index for regression analysis
df['Time_Index'] = np.arange(len(df)) + 1

In [7]:
df

,Month,Sales,Time_Index
0,2019-01-31,412.92,1
1,2019-02-28,341.00,2
2,2019-03-31,324.47,3
3,2019-04-30,320.42,4
4,2019-05-31,277.35,5
5,2019-06-30,192.90,6
6,2019-07-31,259.00,7
7,2019-08-31,266.59,8
8,2019-09-30,297.42,9
9,2019-10-31,341.29,10


In [8]:
# Linear Regression using ols
model = smf.ols(formula='Sales ~ Time_Index', data=df).fit()

In [9]:
# Forecast sales for the next year 12 months
df_forecast = pd.DataFrame({
    'Month': pd.date_range(start='2022-01-01', periods=12, freq='M'),
    'Time_Index': np.arange(37, 49)
})

C:\Users\LENOVO-PC\AppData\Local\Temp\ipykernel_57660\1539330755.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  'Month': pd.date_range(start='2022-01-01', periods=12, freq='M'),


In [10]:
df_forecast

,Month,Time_Index
0,2022-01-31,37
1,2022-02-28,38
2,2022-03-31,39
3,2022-04-30,40
4,2022-05-31,41
5,2022-06-30,42
6,2022-07-31,43
7,2022-08-31,44
8,2022-09-30,45
9,2022-10-31,46


In [11]:
df_forecast['Forecasted_Sales'] = model.predict(df_forecast)

In [12]:
df_forecast

,Month,Time_Index,Forecasted_Sales
0,2022-01-31,37,296.614127
1,2022-02-28,38,296.040716
2,2022-03-31,39,295.467306
3,2022-04-30,40,294.893895
4,2022-05-31,41,294.320485
5,2022-06-30,42,293.747074
6,2022-07-31,43,293.173664
7,2022-08-31,44,292.600253
8,2022-09-30,45,292.026843
9,2022-10-31,46,291.453432


In [13]:
# Calculate the average monthly sales for the three years
avg_monthly_sales = df['Sales'].groupby(df['Month'].dt.month).mean()

In [14]:
avg_monthly_sales

Month
1     407.946667
2     321.450000
3     312.236667
4     287.906667
5     272.630000
6     211.673333
7     249.453333
8     265.293333
9     270.470000
10    321.596667
11    366.603333
12    399.406667
Name: Sales, dtype: float64

In [15]:
# Calculate the overall average sales across all months
overall_avg_sales = df['Sales'].mean()

In [16]:
overall_avg_sales

np.float64(307.22222222222223)

In [17]:
# Calculate the seasonal index for each month
df_forecast['Month_Index'] = df_forecast['Month'].dt.month

In [18]:
df_forecast['Month_Index']

0      1
1      2
2      3
3      4
4      5
5      6
6      7
7      8
8      9
9     10
10    11
11    12
Name: Month_Index, dtype: int32

In [19]:
df_forecast['Seasonal_Index'] = df_forecast['Month_Index'].apply(lambda x: avg_monthly_sales[x] / overall_avg_sales)

In [20]:
df_forecast['Seasonal_Index']

0     1.327855
1     1.046311
2     1.016322
3     0.937128
4     0.887403
5     0.688991
6     0.811964
7     0.863523
8     0.880373
9     1.046788
10    1.193284
11    1.300058
Name: Seasonal_Index, dtype: float64

In [21]:
# Adjust the annual forecast using the seasonal index to get the seasonally adjusted forecast for each month
df_forecast['Seasonally_Adjusted_Forecast'] = df_forecast['Forecasted_Sales'] * df_forecast['Seasonal_Index']

df_forecast[['Month', 'Seasonally_Adjusted_Forecast']]

,Month,Seasonally_Adjusted_Forecast
0,2022-01-31,393.860651
1,2022-02-28,309.750667
2,2022-03-31,300.289888
3,2022-04-30,276.353442
4,2022-05-31,261.180956
5,2022-06-30,202.389078
6,2022-07-31,238.046412
7,2022-08-31,252.666932
8,2022-09-30,257.092405
9,2022-10-31,305.090080


In [22]:
# Choose the window size (e.g., 12 months)
window_size = 12

# Calculate the rolling moving average on the historical data
df['Moving_Avg'] = df['Sales'].rolling(window=window_size).mean()

# The Moving_Avg column will now contain the 12-month moving average of sales.


In [23]:
df

,Month,Sales,Time_Index,Moving_Avg
0,2019-01-31,412.92,1,NaN
1,2019-02-28,341.00,2,NaN
2,2019-03-31,324.47,3,NaN
3,2019-04-30,320.42,4,NaN
4,2019-05-31,277.35,5,NaN
5,2019-06-30,192.90,6,NaN
6,2019-07-31,259.00,7,NaN
7,2019-08-31,266.59,8,NaN
8,2019-09-30,297.42,9,NaN
9,2019-10-31,341.29,10,NaN
